# CatBoost for UK used-car prices
Use **Run All** to compare two CatBoost regressors on the shared training and validation splits. One learns price in pounds and the other learns log price. Both are selected by validation mean absolute error (MAE) in pounds. The previous Random Forest validation MAE was £1,167.55.

Install the packages in the repository's `requirements.txt` before running the notebook. Training progress appears in the notebook and is saved under `outputs/uk-modeling/`.

[UK data and cleaning code](https://github.com/MazenBassem/Used-Car-Price-Prediction-Market-Pattern-Discovery-Data-Mining/tree/afa7ea823febad0e4ebcdf0cf61a9b9cf1c1fb8e)

In [ ]:
import math
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor, Pool
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

# Jupyter can start the kernel from the repository or the notebooks folder.
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

from src.uk_modeling import CATEGORIES, NUMBERS, load_splits

output_dir = project_root / "outputs" / "uk-modeling"
output_dir.mkdir(parents=True, exist_ok=True)
RF_VAL_MAE = 1167.55
PARAMS = dict(depth=8, learning_rate=0.06, loss_function="RMSE", task_type="CPU",
              random_seed=42, thread_count=6, allow_writing_files=False)

## Load the shared splits
Identical vehicle profiles stay together. CatBoost uses the four categorical fields directly and preserves numeric missing values. Only the training and validation partitions are used for model selection.

In [ ]:
def features(frame):
    result = frame[CATEGORIES + NUMBERS].copy()
    result[CATEGORIES] = result[CATEGORIES].fillna("Unknown").astype(str)
    return result

train, valid, test = load_splits()
print(f"Training: {len(train):,} rows | Validation: {len(valid):,} rows")
display(train.head(3))

## Train both candidates
Each boosting iteration adds one tree, up to 5,000. CatBoost prints training and validation metrics every 25 iterations. For both candidates, the displayed validation metric and the early-stopping criterion are absolute error in pounds. Stop after 100 iterations without improvement and retain the best tree count.

In [ ]:
class LogPriceMAE:
    # CatBoost passes log prices here. Convert each side to GBP before scoring.
    def evaluate(self, approxes, target, weight):
        score, total_weight = 0.0, 0.0
        for i in range(len(target)):
            w = 1.0 if weight is None else weight[i]
            score += w * abs(math.exp(approxes[0][i]) - math.exp(target[i]))
            total_weight += w
        return score, total_weight

    def get_final_error(self, error, weight):
        return error / max(weight, 1e-12)

    def is_max_optimal(self):
        return False

class TrainingOutput:
    def __init__(self, file):
        self.file = file

    def write(self, text):
        sys.stdout.write(text)
        self.file.write(text)
        self.flush()

    def flush(self):
        sys.stdout.flush()
        self.file.flush()

def pool(frame, target_type):
    y = frame.price if target_type == "GBP" else np.log(frame.price)
    return Pool(features(frame), y, cat_features=CATEGORIES)

def price(model, frame, target_type):
    raw = model.predict(features(frame))
    return raw if target_type == "GBP" else np.exp(raw)

scores, curves = [], {}
with (output_dir / "catboost_training.log").open("w", encoding="utf-8") as log:
    output = TrainingOutput(log)
    for target_type in ["GBP", "log GBP"]:
        metric = "MAE" if target_type == "GBP" else LogPriceMAE()
        metric_key = "MAE" if target_type == "GBP" else "LogPriceMAE"
        output.write(f"Training target: {target_type} | evaluation metric: MAE in pounds\n")
        model = CatBoostRegressor(**PARAMS, iterations=5000, eval_metric=metric)
        model.fit(pool(train, target_type), eval_set=pool(valid, target_type),
                  early_stopping_rounds=100, use_best_model=True, verbose=25,
                  log_cout=output, log_cerr=output)
        pred = price(model, valid, target_type)
        curves[target_type] = model.get_evals_result()["validation"][metric_key]
        scores.append({"Target": target_type, "Trees": model.tree_count_,
                       "Validation MAE (£)": mean_absolute_error(valid.price, pred),
                       "Validation RMSE (£)": root_mean_squared_error(valid.price, pred)})
        output.write(f"Best checkpoint: {model.tree_count_} trees | validation MAE £{scores[-1]['Validation MAE (£)']:,.2f}\n")
comparison = pd.DataFrame(scores).sort_values("Validation MAE (£)")
display(comparison.round(2))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4), layout="constrained")
for label, values in curves.items():
    ax.plot(range(1, len(values) + 1), values, label=f"Target: {label}")
ax.axhline(RF_VAL_MAE, ls="--", color="gray", label="Random Forest")
ax.set(xlabel="Trees", ylabel="Validation MAE (£)", yscale="log",
       title="Price error as trees are added (log scale)")
ax.legend()
plt.show()

## Refit the winner and evaluate the test split
Use the selected target and tree count to refit on training plus validation, matching the Random Forest refit procedure. The saved bundle includes the fitted regressor and its target transformation.

In [ ]:
best = comparison.iloc[0]
target_type = best["Target"]
final = CatBoostRegressor(**PARAMS, iterations=int(best["Trees"]))
with (output_dir / "catboost_training.log").open("a", encoding="utf-8") as log:
    output = TrainingOutput(log)
    output.write(f"Refitting {target_type} on training + validation (training RMSE in target units)\n")
    final.fit(pool(pd.concat([train, valid], ignore_index=True), target_type),
              verbose=25, log_cout=output, log_cerr=output)
model_path = output_dir / "catboost_price.joblib"
temporary_path = model_path.with_suffix(".tmp")
joblib.dump({"model": final, "target": target_type}, temporary_path)
temporary_path.replace(model_path)

pred = price(final, test, target_type)
result = [mean_absolute_error(test.price, pred), root_mean_squared_error(test.price, pred),
          r2_score(test.price, pred)]
print(f"Test listings: {len(test):,}")
display(pd.DataFrame([result, [1138.48, 1897.52, 0.9633]],
                     index=["CatBoost", "Random Forest (previous run)"],
                     columns=["Test MAE (£)", "Test RMSE (£)", "Test R²"]).round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4), layout="constrained")
bins = ax.hexbin(test.price, pred, gridsize=45, bins="log", mincnt=1, cmap="viridis")
limits = [min(test.price.min(), pred.min()), max(test.price.max(), pred.max())]
ax.plot(limits, limits, "--", color="darkorange", label="Exact prediction")
ax.set(title="Held-out UK listings", xlabel="Listed price (£)", ylabel="Predicted price (£)")
ax.legend()
fig.colorbar(bins, ax=ax, label="Listings per bin")
plt.show()

## Try a car
Edit this example and rerun the cell. Use miles for mileage, litres for engine size, GBP for annual road tax, and UK mpg for fuel economy.

In [ ]:
car = {"brand": "Ford", "model": "Fiesta", "year": 2018, "mileage": 35000,
       "transmission": "Manual", "fuel_type": "Petrol", "engine_size": 1.0,
       "tax": 150, "mpg": 60.1}
bundle = joblib.load(model_path)
estimate = price(bundle["model"], pd.DataFrame([car]), bundle["target"])[0]
print(f"{car['year']} {car['brand']} {car['model']} | estimated UK listing price £{estimate:,.0f}")

In [ ]:
from IPython.display import display, Markdown

display(Markdown("## Visual summary\nThese charts summarize the selected CatBoost model on the held-out test listings. Residual is predicted price minus listed price; positive values mean overprediction."))

actual = test.price.to_numpy()
residual = pred - actual
abs_error = np.abs(residual)

fig, axes = plt.subplots(3, 2, figsize=(14, 14), layout="constrained")

ax = axes[0, 0]
ax.scatter(actual, pred, s=8, alpha=0.18, color="#287c8e", edgecolors="none")
limits = [min(actual.min(), pred.min()), max(actual.max(), pred.max())]
ax.plot(limits, limits, "--", color="#e76f51", label="Exact prediction")
ax.set(xscale="log", yscale="log", xlabel="Listed price (£)", ylabel="Predicted price (£)",
       title="Predicted versus listed price")
ax.legend()

ax = axes[0, 1]
ax.hist(residual, bins=60, color="#4c956c", alpha=0.85)
ax.axvline(0, color="#e76f51", linestyle="--", linewidth=1.5)
ax.set(xlabel="Prediction error (£)", ylabel="Listings", title="Prediction error distribution")

ax = axes[1, 0]
ax.scatter(actual, residual, s=8, alpha=0.16, color="#6c5ce7", edgecolors="none")
ax.axhline(0, color="#e76f51", linestyle="--", linewidth=1.5)
ax.set(xscale="log", xlabel="Listed price (£)", ylabel="Predicted − listed (£)",
       title="Prediction error by listed price")

price_bands = pd.qcut(test.price, q=5, duplicates="drop")
price_mae = pd.DataFrame({"band": price_bands, "abs_error": abs_error}).groupby(
    "band", observed=True).abs_error.mean()
ax = axes[1, 1]
ax.bar([f"£{band.left:,.0f}–£{band.right:,.0f}" for band in price_mae.index],
       price_mae.values, color="#f4a261")
ax.set(xlabel="Listed-price band", ylabel="Mean absolute error (£)", title="MAE by price band")
ax.tick_params(axis="x", rotation=25)

brand_errors = test[["brand"]].copy()
brand_errors["abs_error"] = abs_error
brand_summary = brand_errors.groupby("brand").abs_error.agg(MAE="mean", listings="size")
brand_summary = brand_summary[brand_summary.listings >= 200].nlargest(10, "MAE").sort_values("MAE")
ax = axes[2, 0]
ax.barh(brand_summary.index, brand_summary.MAE, color="#e76f51")
ax.set(xlabel="Mean absolute error (£)", title="Highest-error brands (at least 200 listings)")

importance = pd.Series(final.feature_importances_, index=final.feature_names_).nlargest(10).sort_values()
ax = axes[2, 1]
ax.barh(importance.index, importance.values, color="#577590")
ax.set(xlabel="Feature importance", title="Most influential features")

plt.show()